In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv('../.env')

True

In [2]:
df = pd.read_csv('../data/raw/Chicago_Crimes_2012_to_2017.csv', 
                 low_memory=False)
print(f"Raw records: {len(df):,}")

Raw records: 1,456,714


In [3]:
# Drop rows with missing lat/lon
df = df.dropna(subset=['Latitude', 'Longitude'])

# Filter to valid Chicago bounding box only
# Chicago is roughly between these coordinates
df = df[
    (df['Latitude'] >= 41.6) & (df['Latitude'] <= 42.1) &
    (df['Longitude'] >= -87.95) & (df['Longitude'] <= -87.5)
]

print(f"After cleaning: {len(df):,}")
print(f"Removed: {1456714 - len(df):,} bad rows")

After cleaning: 1,419,554
Removed: 37,160 bad rows


In [21]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['hour']       = df['Date'].dt.hour
df['day_of_week'] = df['Date'].dt.dayofweek   # 0=Monday, 6=Sunday
df['month']      = df['Date'].dt.month
df['year']       = df['Date'].dt.year
df['is_weekend'] = df['day_of_week'].isin([5, 6])

print(df[['Date','hour','day_of_week','month','is_weekend']].head())

                 Date  hour  day_of_week  month  is_weekend
0 2016-05-03 23:40:00    23            1      5       False
1 2016-05-03 21:40:00    21            1      5       False
2 2016-05-03 23:31:00    23            1      5       False
3 2016-05-03 22:10:00    22            1      5       False
4 2016-05-03 22:00:00    22            1      5       False


In [22]:
df_clean = df[[
    'ID', 'Case Number', 'Date', 'Primary Type', 'Description',
    'Location Description', 'Arrest', 'Domestic', 'District',
    'Community Area', 'Year', 'Latitude', 'Longitude',
    'hour', 'day_of_week', 'month', 'is_weekend'
]].copy()

# Fill minor nulls
df_clean['Location Description'] = df_clean['Location Description'].fillna('Unknown')
df_clean['District'] = df_clean['District'].fillna(0).astype(int)

print(df_clean.shape)
print(df_clean.isnull().sum())

(1419554, 17)
ID                       0
Case Number              0
Date                     0
Primary Type             0
Description              0
Location Description     0
Arrest                   0
Domestic                 0
District                 0
Community Area          22
Year                     0
Latitude                 0
Longitude                0
hour                     0
day_of_week              0
month                    0
is_weekend               0
dtype: int64


In [23]:
# Rename columns to match Supabase table schema
df_clean = df_clean.rename(columns={
    'ID': 'id',
    'Case Number': 'case_number',
    'Date': 'crime_date',
    'Primary Type': 'crime_type',
    'Description': 'description',
    'Location Description': 'location_description',
    'Arrest': 'arrest',
    'Domestic': 'domestic',
    'District': 'district',
    'Community Area': 'community_area',
    'Year': 'year',
    'Latitude': 'latitude',
    'Longitude': 'longitude'
})

print(df_clean.columns.tolist())

['id', 'case_number', 'crime_date', 'crime_type', 'description', 'location_description', 'arrest', 'domestic', 'district', 'community_area', 'year', 'latitude', 'longitude', 'hour', 'day_of_week', 'month', 'is_weekend']


In [24]:
df_clean['community_area'] = df_clean['community_area'].fillna(0).astype(int)

print(df_clean['community_area'].isnull().sum())

0


In [25]:
geometry = [Point(xy) for xy in zip(df_clean['longitude'], df_clean['latitude'])]
gdf = gpd.GeoDataFrame(df_clean, geometry=geometry, crs='EPSG:4326')

print(gdf.shape)
gdf.head(2)

(1419554, 18)


,id,case_number,crime_date,crime_type,description,location_description,arrest,domestic,district,community_area,year,latitude,longitude,hour,day_of_week,month,is_weekend,geometry
0,10508693,HZ250496,2016-05-03 23:40:00,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,True,True,10,29,2016,41.864073,-87.706819,23,1,5,False,POINT (-87.70682 41.86407)
1,10508695,HZ250409,2016-05-03 21:40:00,BATTERY,DOMESTIC BATTERY SIMPLE,RESIDENCE,False,True,3,42,2016,41.782922,-87.604363,21,1,5,False,POINT (-87.60436 41.78292)


In [26]:
df_clean.to_csv('../data/processed/crimes_cleaned.csv', index=False)
print("Saved to data/processed/crimes_cleaned.csv")

Saved to data/processed/crimes_cleaned.csv


In [27]:
DATABASE_URL = os.getenv('DATABASE_URL')
engine = create_engine(DATABASE_URL)

# Load in chunks — 1.4M rows at once will crash
chunk_size = 10000
total = len(gdf)

for i in range(0, total, chunk_size):
    chunk = gdf.iloc[i:i+chunk_size]
    chunk.drop(columns='geometry').to_sql(
        'crime_incidents', 
        engine, 
        if_exists='append', 
        index=False,
        method='multi',
        chunksize=1000
    )
    print(f"Loaded {min(i+chunk_size, total):,} / {total:,} rows", end='\r')

print("\nDone! All data loaded into Supabase.")

Loaded 1,419,554 / 1,419,554 rows
Done! All data loaded into Supabase.
